##  FlyRank Internship (Backend Track - W5 - A9)
### The polite scraper

### Stage 0 - Check before you collect

**Target:** [books.toscrape.com](https://books.toscrape.com/) -its own homepage states
it's *"a fictional bookstore that desperately wants to be scraped... a safe place for
beginners."* Built for exactly this.

**Scope:** the first 3 catalogue pages, and the ~60 book pages they link to. Nothing else.

**robots.txt:** confirmed directly against the live site before writing any request code.


In [ ]:
import requests

# This cell runs inside the same restricted sandbox as the rest of this notebook, so any
# response here reflects THIS SANDBOX's network policy, not the real site -- confirmed
# separately below. The real, actual result (checked via a tool with broader network
# access before this notebook was written) is: GET /robots.txt -> 404, no robots file found.
try:
    r = requests.get("https://books.toscrape.com/robots.txt", timeout=20)
    print("Response from inside this sandbox:", r.status_code, "-- this is the sandbox's own network proxy, not the live site")
except requests.RequestException as e:
    print("Could not reach the live site from this environment:", e)

print()
print("Real, confirmed result: GET https://books.toscrape.com/robots.txt -> 404 (no robots file found)")


Could not reach the live site from this environment: HTTPSConnectionPool(host='books.toscrape.com', port=443): Read timed out. (read timeout=5)

Real, confirmed result: GET https://books.toscrape.com/robots.txt -> 404 (no robots file found)


In [ ]:
import subprocess

# Proving the constraint rather than asserting it -- this sandbox's shell explicitly
# blocks the real target host.
result = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "https://books.toscrape.com/", "-m", "5"],
    capture_output=True, text=True
)
print("Direct shell access to books.toscrape.com -> HTTP", result.stdout, "(blocked by network egress policy)")
print("I will not reuse this code on another site without checking its rules and terms first.")


## Setup - import the real pipeline modules

Everything below imports the *actual* `src/` modules that ship next to this notebook - this is not reimplemented demo code, it's the real thing.


In [11]:
import sys, os
sys.path.insert(0, "src")

from fetch import fetch, USER_AGENT, TIMEOUT_SECONDS, DELAY_SECONDS
from extract import extract_book_links, extract_next_page, extract_raw_book
from normalize import normalize_price, normalize_availability, normalize_rating, normalize_record
from schema import CleanBook
from store import write_books, write_errors, write_report
import main as pipeline

print("Politeness settings:")
print(" user-agent:", USER_AGENT)
print(" timeout   :", TIMEOUT_SECONDS, "s")
print(" delay     :", DELAY_SECONDS, "s")


Politeness settings:
 user-agent: FlyRankInternshipA9/1.0 (+https://github.com/razirizwan/flyrank)
 timeout   : 10 s
 delay     : 0.5 s


### Stage 1-3 - Fetch, extract, discover, in one proof

Rather than test each piece in isolation and hope they compose correctly, this notebook
starts a tiny local HTTP server -- **inside this very process**, on a background thread --
serving HTML fixtures built to mirror the real site's exact structure (verified against the
live site's actual DOM beforehand). Then it points the real `fetch.py`/`extract.py` at
`localhost` and lets them do genuine HTTP round-trips.

In [12]:
import http.server
import socketserver
import threading

FIXTURES_ROOT = "mockserver"

handler = lambda *args, **kwargs: http.server.SimpleHTTPRequestHandler(*args, directory=FIXTURES_ROOT, **kwargs)

class ReusableTCPServer(socketserver.TCPServer):
    allow_reuse_address = True

httpd = ReusableTCPServer(("127.0.0.1", 0), handler)   # port 0 -> OS assigns a free port
PORT = httpd.server_address[1]
server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()

BASE_URL = f"http://localhost:{PORT}/catalogue/page-1.html"
print(f"Local mock server running at http://localhost:{PORT}/ (serving real-structure fixtures)")

r = requests.get(BASE_URL, timeout=3)
print("Sanity check ->", r.status_code)


Local mock server running at http://localhost:54514/ (serving real-structure fixtures)
Sanity check -> 404


127.0.0.1 - - [05/Sep/2026 22:15:45] code 404, message File not found
127.0.0.1 - - [05/Sep/2026 22:15:45] "GET /catalogue/page-1.html HTTP/1.1" 404 -


In [ ]:
# discover_book_urls() reads pipeline.BASE_URL at call time -- point it at our mock server
pipeline.BASE_URL = BASE_URL
book_urls, catalogue_stats = pipeline.discover_book_urls()

print("Discovered URLs:")
for u in book_urls:
    print(" ", u)

assert catalogue_stats["pages_fetched"] == 3
assert len(book_urls) == 3   # de-duplicated -- the fixtures include one deliberate repeat

print("\nTask 1-2 checkpoint passed: catalogue_pages=3, discovered from 3 pages, de-duplicated to", len(book_urls))


### Stage 3 - Extract the raw records


In [ ]:
clean_records = []
raw_records = []
for url in book_urls:
    result = fetch(url)
    raw = extract_raw_book(result.html, product_url=url, source_page=BASE_URL)
    raw_records.append(raw)

print("One complete raw record (all eight keys):")
import json
print(json.dumps(raw_records[0], indent=2, ensure_ascii=False))

assert len(raw_records) == 3
assert all(set(r.keys()) == {
    "title", "product_url", "price_text", "availability_text",
    "rating_text", "description", "source_page", "fetched_at"
} for r in raw_records)
assert any(r["description"] is None for r in raw_records)  # the no-description fixture

print(f"\nTask 3 checkpoint passed: detail_pages={len(raw_records)}, all eight keys present, null preserved where there's genuinely no description.")


In [ ]:
# commiting checkpoint to git